# Phase-1 Feature Analytics Run + Results Review

This notebook helps you run and review first results for:
- prediction dump build from validation data + true Two Tower scoring
- true Two Tower permutation importance (`two_tower_true_rescore`)
- residual dataset and weak slices
- true re-score guard report

Execution sequence:
1. bootstrap manifest + feature catalog
2. build `validation_predictions.parquet`
3. validate contracts
4. run Phase-1 analytics
5. review outputs

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import yaml

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CONFIG_PATH = ROOT / "configs" / "analytics.yaml"
TRAIN_CFG_PATH = ROOT.parent / "Two_tower" / "configs" / "train.yaml"

print("ROOT:", ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("TRAIN_CFG_PATH:", TRAIN_CFG_PATH)

# Optional: install this repo in editable mode when running in fresh SageMaker kernel.
# !{sys.executable} -m pip install -e {ROOT}

In [ ]:
# Bootstrap `inputs/model_manifest.json` and `inputs/feature_catalog.parquet` from Two Tower train config.
# This removes manual setup for first run.

inputs_dir = ROOT / "inputs"
inputs_dir.mkdir(parents=True, exist_ok=True)

with open(TRAIN_CFG_PATH, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

features = train_cfg["features"]
user_features = list(features.get("user_feature_cols") or [])
client_features = list(features.get("client_feature_cols") or [])
user_multi = set(features.get("user_multi_cols") or [])
client_multi = set(features.get("client_multi_cols") or [])

rows = []
for c in user_features:
    rows.append({
        "feature_name": c,
        "tower": "user",
        "feature_type": "multi" if c in user_multi else "other",
        "is_multi": c in user_multi,
    })
for c in client_features:
    rows.append({
        "feature_name": c,
        "tower": "client",
        "feature_type": "multi" if c in client_multi else "other",
        "is_multi": c in client_multi,
    })

feature_catalog = pd.DataFrame(rows).drop_duplicates(subset=["feature_name"]).sort_values("feature_name")
feature_catalog_path = inputs_dir / "feature_catalog.parquet"
feature_catalog.to_parquet(feature_catalog_path, index=False)

manifest = {
    "model_id": "two_tower_mlp",
    "model_version": "v1",
    "trained_at": "2026-05-01T00:00:00Z",
    "label_column": features.get("label_col", "label"),
    "prediction_schema_version": "1.0.0",
    "feature_catalog_version": "1.0.0",
    "artifact_uris": {
        "artifacts_base": train_cfg["paths"]["artifacts_base"],
    },
}
manifest_path = inputs_dir / "model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Wrote:", feature_catalog_path)
print("Wrote:", manifest_path)
print("Feature rows:", len(feature_catalog))

In [ ]:
# Build prediction dump, validate contracts, and run Phase-1 pipeline.

build_cmd = [sys.executable, "-m", "pipelines.build_prediction_dump", "--config", str(CONFIG_PATH)]
validate_cmd = [sys.executable, "-m", "pipelines.validate_contracts", "--config", str(CONFIG_PATH)]
run_cmd = [sys.executable, "-m", "pipelines.run_mvp", "--config", str(CONFIG_PATH)]

print("Running:", " ".join(build_cmd))
subprocess.run(build_cmd, cwd=str(ROOT), check=True)

print("Running:", " ".join(validate_cmd))
subprocess.run(validate_cmd, cwd=str(ROOT), check=True)

print("Running:", " ".join(run_cmd))
subprocess.run(run_cmd, cwd=str(ROOT), check=True)

In [ ]:
# Load and display outputs.

out_root = ROOT / "outputs"
pi_path = out_root / "feature_importance" / "permutation_importance.csv"
guard_path = out_root / "feature_importance" / "true_rescore_guard_report.json"
residual_slices_path = out_root / "residual_analysis" / "residual_slices.csv"

pi_df = pd.read_csv(pi_path)
residual_slices_df = pd.read_csv(residual_slices_path)

guard = json.loads(guard_path.read_text(encoding="utf-8"))

print("Top 20 features by AUC drop")
display(pi_df.sort_values("auc_drop", ascending=False).head(20))

print("\nTop 20 weak slices by avg residual")
display(residual_slices_df.sort_values("avg_residual_abs", ascending=False).head(20))

print("\nGuard report summary")
display(pd.DataFrame([
    {
        "mode": guard.get("mode"),
        "used_features": guard.get("used_feature_cols_count"),
        "ignored_requested": guard.get("ignored_requested_feature_cols_count"),
        "missing_scorable_features": guard.get("missing_scorable_feature_cols_count"),
        "missing_client_embedding_count": guard.get("missing_client_embedding_count"),
        "missing_client_embedding_ratio": guard.get("missing_client_embedding_ratio"),
    }
]))

print("\nIgnored requested features (first 50):")
print((guard.get("ignored_requested_feature_cols") or [])[:50])